# RoBERTa-LSTM_training.ipynb

**Author:** Zane Zhang

**Date:** 2025/07/06 23:16

**Description:**

This script aims to construct the RoBERTa-LSTM model and its training. Why we use this model to prepare for misinfo detection is the MentalLLaMa and RoBERTa both have a maximum length for input text. Inspired by Arora et al. (2025), we adapt RoBERTa-LSTM as the target model.
    
In this script, we first prepare the training and testing dataset. Then we run each model respectively with the category weighting system. Based on the comparison results, the RoBERTa-LSTM performanced better than other models. Thus we use the RoBERT-LSTM model to label the remaining data.

Note: All experiments are conducted on Colab with A100.


In [ ]:
!pip install --upgrade scikit-learn datasets pyod

In [ ]:
import pandas as pd
import random
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
     RobertaTokenizer, RobertaForSequenceClassification, AutoModel, Trainer, TrainingArguments, DataCollatorWithPadding
)
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from pyod.models.ecod import ECOD
from sklearn.feature_extraction.text import TfidfVectorizer

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_excel("drive/MyDrive/DMisinfo/manual_annotation.xlsx")
df['text'] = df['Title_normalized'] + " " + df['Sentence_normalized']
df = df[['text', 'Label']]  # keep only required colums

# Split dataset
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Label'], random_state=42)
# Covert to HuggingFace datasets
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
# HuggingFace requires lowercase by default
train_ds = train_ds.rename_column("Label", "labels")
test_ds = test_ds.rename_column("Label", "labels")

In [ ]:
# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# Load tokenizer
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_fn(example):
  return tokenizer(example['text'], truncation=True)

train_tokenized = train_ds.map(tokenize_fn, batched=True)
test_tokenized = test_ds.map(tokenize_fn, batched=True)

In [ ]:
# Compute class weights
labels = train_tokenized['labels']
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cuda')

First, we train the only RoBERTa model. In this process we use category weighting system to balance the number of Misinfo and Non-Misinfo.

In [ ]:
# Custom Trainer with weights
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Load model and collator
roberta_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2).cuda()
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/DMisinfo/roberta_model_weighted",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=9,
    weight_decay=0.01,
    logging_dir="/content/logs",
    save_total_limit=1,
    report_to="none",
    seed=42,                          # ensure consistent behavior
    data_seed=42                      # ensure same shuffling of data
)

# Trainer initialization and training
roberta_trainer = WeightedTrainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator
)

roberta_trainer.train()

# Evaluation
roberta_preds = roberta_trainer.predict(test_tokenized)
y_true_roberta = roberta_preds.label_ids
# y_pred_roberta = np.argmax(roberta_preds.predictions, axis=1)
probs_roberta = torch.softmax(torch.tensor(roberta_preds.predictions), dim=1).numpy()
y_pred_roberta = (probs_roberta[:, 1] >= 0.4).astype(int)

f1_macro_roberta = f1_score(y_true_roberta, y_pred_roberta, average='macro')
f1_pos_roberta = f1_score(y_true_roberta, y_pred_roberta, pos_label=1)
precision_roberta = precision_score(y_true_roberta, y_pred_roberta, pos_label=1)
recall_roberta = recall_score(y_true_roberta, y_pred_roberta, pos_label=1)

In [ ]:
# 混淆矩阵计算与可视化
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 计算混淆矩阵
cm = confusion_matrix(y_true_roberta, y_pred_roberta)

# 创建可视化图像
plt.figure(figsize=(6, 6))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    square=True,
    linewidths=0.5,
    linecolor='white',
    xticklabels=True,  # 不显示刻度文本
    yticklabels=True,
    cbar_kws={"shrink": 0.7}
)
plt.xlabel("Predicted label", fontsize=12)
plt.ylabel("True label", fontsize=12)
plt.tight_layout()
plt.show()

Second, we adapt only the LSTM.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3, num_labels=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embedding(input_ids)
        lstm_out, _ = self.lstm(x)
        pooled = lstm_out[:, -1, :]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(weight=class_weights)
            loss = loss_fct(logits, labels)
        return {'logits': logits, 'loss': loss} if loss is not None else {'logits': logits}

# 用 tokenizer 词表
vocab_size = tokenizer.vocab_size
lstm_model = LSTMClassifier(vocab_size).cuda()

# 需要将 input_ids 作为输入，attention_mask 可忽略
def lstm_data_format(example):
    return {'input_ids': example['input_ids'], 'labels': example['labels']}
train_lstm = train_tokenized.map(lstm_data_format, batched=True)
test_lstm = test_tokenized.map(lstm_data_format, batched=True)

lstm_trainer = Trainer(
    model=lstm_model,
    args=training_args,
    train_dataset=train_lstm,
    eval_dataset=test_lstm,
    tokenizer=tokenizer,
    data_collator=data_collator
)
lstm_trainer.train()
lstm_preds = lstm_trainer.predict(test_lstm)
y_true_lstm = lstm_preds.label_ids
# y_pred_lstm = np.argmax(lstm_preds.predictions, axis=1)
probs_lstm = torch.softmax(torch.tensor(lstm_preds.predictions), dim=1).numpy()
y_pred_lstm = (probs_lstm[:, 1] >= 0.4).astype(int)

f1_macro_lstm = f1_score(y_true_lstm, y_pred_lstm, average='macro')
f1_pos_lstm = f1_score(y_true_lstm, y_pred_lstm, pos_label=1)
precision_lstm = precision_score(y_true_lstm, y_pred_lstm, pos_label=1)
recall_lstm = recall_score(y_true_lstm, y_pred_lstm, pos_label=1)

In [ ]:
# 混淆矩阵计算与可视化
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 计算混淆矩阵
cm = confusion_matrix(y_true_lstm, y_pred_lstm)

# 创建可视化图像
plt.figure(figsize=(6, 6))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    square=True,
    linewidths=0.5,
    linecolor='white',
    xticklabels=True,  # 不显示刻度文本
    yticklabels=True,
    cbar_kws={"shrink": 0.7}
)
plt.xlabel("Predicted label", fontsize=12)
plt.ylabel("True label", fontsize=12)
plt.tight_layout()
plt.show()

Third, we construct hybrid model, RoBERTa-LSTM which weight is also considered.

In [ ]:
# Hybrid Model: RoBERTa + LSTM
class RobertaLSTMClassifier(nn.Module):
    def __init__(self, roberta_model_name, hidden_dim=256, lstm_layers=1, dropout=0.3, num_labels=2):
        super(RobertaLSTMClassifier, self).__init__()
        self.roberta = AutoModel.from_pretrained(roberta_model_name)
        self.lstm = nn.LSTM(input_size=self.roberta.config.hidden_size,
                            hidden_size=hidden_dim,
                            num_layers=2,
                            batch_first=True,
                            bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(sequence_output)
        pooled = lstm_out[:, -1, :]  # use the output at the last time step
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(weight=class_weights)
            loss = loss_fct(logits, labels)
        return {'logits': logits, 'loss': loss} if loss is not None else {'logits': logits}

model = RobertaLSTMClassifier(model_name)
model.cuda()
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/DMisinfo/roberta_lstm_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=9,
    weight_decay=0.01,
    logging_dir="/content/logs",
    save_total_limit=1,
    seed=42,
    data_seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

# Evaluation
preds_output = trainer.predict(test_tokenized)
y_true = preds_output.label_ids
# y_pred = np.argmax(preds_output.predictions, axis=1)
# change the threshold to improve recall
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1).numpy()
y_pred = (probs[:, 1] >= 0.4).astype(int)

f1_macro = f1_score(y_true, y_pred, average='macro')
f1_pos = f1_score(y_true, y_pred, pos_label=1)
precision = precision_score(y_true, y_pred, pos_label=1)
recall = recall_score(y_true, y_pred, pos_label=1)

In [ ]:
# 混淆矩阵计算与可视化
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 计算混淆矩阵
cm = confusion_matrix(y_true, y_pred)

# 创建可视化图像
plt.figure(figsize=(6, 6))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    square=True,
    linewidths=0.5,
    linecolor='white',
    xticklabels=True,  # 不显示刻度文本
    yticklabels=True,
    cbar_kws={"shrink": 0.7}
)
plt.xlabel("Predicted label", fontsize=12)
plt.ylabel("True label", fontsize=12)
plt.tight_layout()
plt.show()

Last, we test effectiveness of our model with ECOD algorithm.

In [ ]:
# ECOD TF-IDF
vectorizer = TfidfVectorizer(max_features=512)
X_test = vectorizer.fit_transform(test_df['text']).toarray()

ecod = ECOD()
ecod.fit(X_test)
ecod_scores = ecod.decision_scores_
# Taking the top 20% of scores as anomalies
threshold = np.percentile(ecod_scores, 80)
y_pred_ecod = (ecod_scores >= threshold).astype(int)

f1_macro_ecod = f1_score(y_true, y_pred_ecod, average='macro')
f1_pos_ecod = f1_score(y_true, y_pred_ecod, pos_label=1)
precision_ecod = precision_score(y_true, y_pred_ecod, pos_label=1)
recall_ecod = recall_score(y_true, y_pred_ecod, pos_label=1)

In [ ]:
# Table output
results = pd.DataFrame({
    'Model': ['RoBERTa', 'LSTM', 'RoBERTa-LSTM', 'ECOD'],
    'F1-Macro': [f"{f1_macro_roberta:.4f}", f"{f1_macro_lstm:.4f}", f"{f1_macro:.4f}", f"{f1_macro_ecod:.4f}"],
    'F1-Pos': [f"{f1_pos_roberta:.4f}", f"{f1_pos_lstm:.4f}", f"{f1_pos:.4f}", f"{f1_pos_ecod:.4f}"],
    'Prec': [f"{precision_roberta:.4f}", f"{precision_lstm:.4f}", f"{precision:.4f}", f"{precision_ecod:.4f}"],
    'Rec': [f"{recall_roberta:.4f}", f"{recall_lstm:.4f}", f"{recall:.4f}", f"{recall_ecod:.4f}"]
})

print('\n=== Model Comparison ===')
print(results.to_markdown(index=False))